# **walter**

## **Project Setup**

In [13]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path
import subprocess
import getpass

IN_COLAB = "google.colab" in sys.modules

REPO_NAME = "walter"
GIT_BRANCH = "main"
REPO_PATH = Path("/content") / REPO_NAME


def get_tokens():
    github_token = os.getenv("GITHUB_TOKEN") or getpass.getpass("GitHub token: ")
    return github_token


def install_core_ml_stack():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "transformers==4.44.2",
            "accelerate==0.33.0",
            "pyarrow",
        ],
        check=True,
    )


def setup_repo(github_token):
    os.chdir("/content")

    repo_url = f"https://{github_token}@github.com/Mango-Cats/{REPO_NAME}.git"

    if REPO_PATH.exists():
        os.chdir(REPO_PATH)
        subprocess.run(["git", "fetch", "origin"], check=True)
        subprocess.run(["git", "reset", "--hard", f"origin/{GIT_BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", repo_url], check=True)
        os.chdir(REPO_PATH)

    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)


if IN_COLAB:
    print("Colab detected")

    github_token = get_tokens()

    install_core_ml_stack()
    setup_repo(github_token)

    os.chdir(REPO_PATH)
    print("Project root:", REPO_PATH)

RES_DIR = "results/"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Device:", DEVICE)

Torch: 2.11.0+cpu
Device: cpu


## **Nomenclature and Terminologies**

The dataset $\mathcal{D}_{\text{raw}}$ (represented as `D_raw` in the source code) refers to the raw Philippine human-drug registry, which is freely available as a `.csv` file at [https://verification.fda.gov.ph/drug_productslist.php](https://verification.fda.gov.ph/drug_productslist.php).

The intermediate dataset, $\mathcal{D}_{\text{clean}}$ (`D_clean`), is the result of passing $\mathcal{D}_{\text{raw}}
$ through the preprocessing pipeline. 

The final datasets, $\mathcal{D}_{\text{train}}$ (`D_train`) and $\mathcal{D}_{\text{test}}$ (`D_test`), are used to train and test a weighted sum of similarity measures via a genetic algorithm. These consist of three columns: an ordered pair of drugs formed from the cleaned registry, such that every pair $(x, y) \in \mathcal{D}_{\text{clean}} \times \mathcal{D}_{\text{clean}}$, followed by their label: `p` for positive (LASA) and `u` for noise (unlabeled). These two datasets are derived from an 80-20 split of the union of two disjoint subsets:

*   **$P \subset (\mathcal{D}_{\text{train}} \cup \mathcal{D}_{\text{test}})$** (`P`) is the set of known positives, consisting of ordered drug pairs manually verified as LASA.
*   **$U \subset (\mathcal{D}_{\text{train}} \cup \mathcal{D}_{\text{test}})$** (`U`) is the unlabeled noise set, consisting of randomly paired drugs from $\mathcal{D}_{\text{clean}}$. $U$ acts as the noise class (where true labels are unknown), meaning it may contain undetected LASA pairs.

Furthermore, it is established that $|U| \gg |P|$, $P \cap U = \emptyset$, $P \cup U = \mathcal{D}_{\text{train}} \cup \mathcal{D}_{\text{test}}$, and $\mathcal{D}_{\text{test}} \cap \mathcal{D}_{\text{train}} = \emptyset$.

## **Preprocessing**

The first step is to preprocess (load, validate, clean) the FDA human drug registry dataset to construct our $\mathcal{D}_\text{clean}$ dataset.

Ensure that the FDA human drug registry dataset exists anywhere starting from the root folder and has the same filename defined by `PRIMARY_FNAME`.

The code for this section is located at [`/src/preprocessing.py`](/src/preprocessing.py).

The function `master_maker` is the coordinator function that performs data loading, validation, cleaning, and reporting. 

In [15]:
import pandas as pd
import src.preprocessing as pre

skip = True

if skip:
    filename = Path(pre.CLEANED_FNAME)
    D_clean = pd.read_parquet(filename)
else:
    D_clean = pre.master_maker(sort=True, save=True)

Let's look at the info of the dataset.

In [16]:
display(D_clean.head())
display(D_clean.shape)

,Brand Name
0,0.9% NaCl-Sapher
1,0.9% Sodchlorsaph
2,1 Ceeplus
3,1000Vc
4,2-Gen


(22838, 1)

Finally, let's look at a slice of 10 entries in the dataset by using the `get_rand_entries()` function.

In [17]:
display(pre.get_rand_entries(df=D_clean, count=10))

,Brand Name
3102,Broxan 30
3103,Broxan PSF
3104,Broxifil
3105,Broxil-M
3106,Broxilex
3107,Broxin
3108,Broxitrol
3109,Broxoflem
3110,BROXOLEX
3111,Broxolvan


## **True LASA Pairs**

Now that we have the $\mathcal{D}_\text{clean}$ we can now proceed with constructing $\mathcal{D}_\text{train}$. We will prioritize constructing the subset of true LASA pairs, or the set $P$.

The code for this section is located at [`/src/proposer/`](/src/proposer/).

### **Local Models**

In [ ]:
from pandas import DataFrame
from src.proposer.inference import load_inference, LocalModel, run_inference

ITERATION_COUNT = 400
OUTPUT = RES_DIR + "lasa_run.json"

result = run_inference(
    output_path=OUTPUT,
    D_clean=D_clean,
    model_choice=LocalModel.QWEN3_1_7B,
    iterations=ITERATION_COUNT,
)
P: DataFrame = load_inference(OUTPUT)

<walter> Iteration 1:
	Selected Drug Name: Dapafloxa-10
	Proposed Confusibles: dapag, dapafoz, dafoxa, dapator-10, dapazox 10, bac-10, dapa-z, iflox, dapasaph-10, dapastal-10

<walter> Iteration 2:
	Selected Drug Name: MD-Uretic
	Proposed Confusibles: m&d, metmic, neotic, uket, q-retic, antica, cetil, d-cure, detoc, ivic

<walter> Iteration 3:
	Selected Drug Name: Nanomet
	Proposed Confusibles: vancomet, unomet, dazomet, janumet, nanoxel, novamet, sanomed, enom, danoxime, lanzomep

<walter> Iteration 4:
	Selected Drug Name: Hydrocortibas
	Proposed Confusibles: hydrocortisone sodium succinate, hydrocortisone, hydr+, edrocort, drocort, procor, zocort, hydrocortef, hydrax, hydrex

<walter> Iteration 5:
	Selected Drug Name: Moxibac
	Proposed Confusibles: moxibac 457, moxibam, moximac, myox, maxilac, moxiana, moxicam, moxitam, topibac, meoxiclav-ds

<walter> Iteration 6:
	Selected Drug Name: Finox
	Proposed Confusibles: fusinox, aeruginox, aluminox, ginox, platinox, saphrinox, terbinox, zef

In [ ]:
display(P.head())
display(P.shape)

,Brand Name,Confusible
0,Meno-Vita,neo-vita
1,Meno-Vita,e-vita
2,Meno-Vita,f-vita
3,Meno-Vita,z-vita
4,Meno-Vita,leginsol-vita


(50, 2)

## **Noise Pairs**

Now that we have $P$, we can now complete constructing $\mathcal{D}_\text{train}$ by constructing the set $U$ or the unlabeled noise set.

The default value of `NOISE_COUNT` is 400 so that the resulting number of noise entries is comparable to previous literature (this value produces 159,600 noise drug pairs).

The code for this section is located at [`/src/noise.py`](/src/noise.py).

In [ ]:
import src.noise as noise

NOISE_COUNT = 400

U = noise.make_noise(fda_df=D_clean, true_df=P, n=NOISE_COUNT)

In [ ]:
display(U.head())
display(U.shape)

,Brand Name,Confusible
0,Neuro-Ace,Fobil
1,Neuro-Ace,Pure-Cee'E
2,Neuro-Ace,Airokast
3,Neuro-Ace,Brexemes
4,Neuro-Ace,Primudine


(159600, 2)

## **Assembling**

Now that both subsets are complete. Assembling $\mathcal{D}_\text{train}$ is simply a concatenation of $P$ and $U$. 

In [ ]:
from src.dataset import prepare_and_save_datasets

D_train, D_test = prepare_and_save_datasets(P, U, RES_DIR)

<walter> Successfully saved split datasets:
	- Train: results/train_walter.csv, results/train_walter.parquet
	- Test:  results/test_walter.csv, results/test_walter.parquet


In [ ]:
display(D_train.head())
display(D_train.shape)

,Brand Name,Confusible,label
17701,Zelapine,Proivig,u
144515,Zelex,Trane-West,u
44575,Tigemac 90,Rapcenate,u
17291,Sapholic-5,Mentopas Cool,u
20369,Gastrec 20,Nasamed,u


(127720, 3)

In [ ]:
display(D_test.head())
display(D_test.shape)

,Brand Name,Confusible,label
6307,Exzomel,Hydrocortef,u
132788,Valabin 500,Ivtig,u
8569,Salumed,Rythma,u
144675,Zelex,Mucofar,u
24149,Zithrotech,Amtacil-WFI,u


(31930, 3)